In [1]:
from numpy import array
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
import tensorflow as tf
import numpy as np
import pandas as pd

In [2]:
#carrega o modelo de uma praia específica
#cidade='IGUAPE'
#praia = 'JURÉIA'
cidade='ILHA COMPRIDA'
praia = 'PRAINHA (BALSA)'
model = tf.keras.models.load_model(f'model{cidade}-{praia}.h5')
model.summary()

Model: "sequential_5"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
lstm_10 (LSTM)               (None, 4, 104)            44096     
_________________________________________________________________
lstm_11 (LSTM)               (None, 104)               86944     
_________________________________________________________________
dense_5 (Dense)              (None, 1)                 105       
Total params: 131,145
Trainable params: 131,145
Non-trainable params: 0
_________________________________________________________________


In [3]:
#carrega as medições da praia
parser = (lambda x:datetime.datetime.strptime(x, '%Y.%m.%d')) 
#os dados de sp_beache_update.csv devem ser atualizados semanalmente
#carrega os dados de sp_beache_update.csv
df = pd.read_csv('sp_beaches_update.csv', parse_dates=['Date'])
#organiza por data
df = df.sort_values(by=['Date'])
#se houver alguma medição em branco será removida
df=df.loc[~df['Enterococcus'].isnull()]
#seleciona somente as medições da praia especificada
df=df.loc[df['City']==cidade].loc[df['Beach']==praia][['Date','Enterococcus']]
#formata time series
df.columns = ['ds', 'y']
df.set_index('ds', inplace=True)
df

,y
ds,
2012-01-03,5
2012-01-08,1
2012-01-15,620
2012-01-22,1
2012-01-29,1
...,...
2019-11-03,47
2019-12-01,3
2020-01-05,49


In [4]:
#retorna somente as 4 ultimas medições para praias com medições mensais
ultimasMedicoes = df.iloc[-4:].values
#formata ultimas medições
ultimasMedicoes = ultimasMedicoes.reshape((ultimasMedicoes.shape[1], ultimasMedicoes.shape[0], 1))
ultimasMedicoes

array([[[ 3],
        [49],
        [88],
        [88]]], dtype=int64)

In [5]:
#preve próxima medicao
yhat = model.predict(ultimasMedicoes, verbose=1)
round(yhat[0,0])

1/1 [==============================] - 1s 581ms/sample


7